# 01 - Transformer + Token Classification (Khôi)

Baseline hiện đại, đơn giản nhất: backbone Transformer + linear token classifier, không dùng CRF.

In [ ]:
from pathlib import Path

CANDIDATES = [
    Path.cwd(),
    Path('/content/NLP-test'),
    Path('/content/drive/MyDrive/NLP-test'),
    Path('/Users/kodomotachi/specialist/NLP-test'),
]
ROOT = next((path for path in CANDIDATES if (path / 'src').exists()), None)
assert ROOT is not None, 'Cannot find project root. In Colab, upload/copy NLP-test to /content/NLP-test or /content/drive/MyDrive/NLP-test.'
%cd $ROOT

In [ ]:
import importlib
import src.deep_ner_models as deep_ner_models
importlib.reload(deep_ner_models)
from src.deep_ner_models import load_jsonl, build_label_map, train_hf_token_classifier

train_rows = load_jsonl('data/processed/b_unique.jsonl')
valid_rows = load_jsonl('data/processed/b_valid.jsonl')
test_rows = load_jsonl('data/processed/b_test.jsonl')
label2id, id2label = build_label_map(train_rows + valid_rows + test_rows)
len(train_rows), len(valid_rows), len(test_rows), len(label2id), label2id

## Config

Cấu hình hiện tại là full train cho Colab GPU: dùng toàn bộ dataset, 3 epoch, max length 384. Nếu Colab hết VRAM, giảm `BATCH_SIZE` xuống 4 hoặc 2.

In [ ]:
LIMIT_TRAIN = None
LIMIT_EVAL = None
EPOCHS = 3
BATCH_SIZE = 8
MAX_LENGTH = 384

train_run = train_rows[:LIMIT_TRAIN] if LIMIT_TRAIN else train_rows
valid_run = valid_rows[:LIMIT_EVAL] if LIMIT_EVAL else valid_rows
test_run = test_rows[:LIMIT_EVAL] if LIMIT_EVAL else test_rows

experiments = [
    {'name': 'khai_bert_token_cls', 'model_name': 'bert-base-uncased'},
    {'name': 'khai_roberta_token_cls', 'model_name': 'roberta-base'},
    {'name': 'khai_xlmroberta_token_cls', 'model_name': 'xlm-roberta-base'},
]

In [ ]:
results = []
for exp in experiments:
    print('Running', exp)
    result = train_hf_token_classifier(
        name=exp['name'],
        model_name=exp['model_name'],
        train_rows=train_run,
        valid_rows=valid_run,
        test_rows=test_run,
        label2id=label2id,
        id2label=id2label,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        max_length=MAX_LENGTH,
    )
    results.append(result)
    print(result['valid'], result['test'])
results